# Panda power three bus problem.

In [ ]:
import pandapower as pp

# Create empty net
# https://pandapower.readthedocs.io/en/stable/elements/empty_network.html

net = pp.create_empty_network() 


In [ ]:
# Create 3 buses, b1,b2,b3
# https://pandapower.readthedocs.io/en/stable/elements/bus.html
V_nom_kV = 220

# A bus is a node of the network: a point where lines, loads and generators meet.
# All three buses are on the same 220 kV voltage level, so they share vn_kv.
# pp.create_bus returns the new bus's index (0, 1, 2), which we keep in b1, b2, b3.
b1 = pp.create_bus(net, vn_kv=V_nom_kV,  name="Bus 1")
b2 = pp.create_bus(net, vn_kv=V_nom_kV,  name="Bus 2")
b3 = pp.create_bus(net, vn_kv=V_nom_kV,  name="Bus 3")


# inspect the bus objects
net.bus

In [ ]:
###### create bus elements

# BUS 1 
# Create a swing (also known as a slack) bus, V=1.05 py Angle =0
# Note: Its called an external grid in PP
# https://pandapower.readthedocs.io/en/stable/elements/ext_grid.html

pp.create_ext_grid(net, bus=b1, vm_pu=1.05, name="Swing bus")


# BUS 2 
# Create a load bus, P=256.6MW, Q=110.2 Mvar
# https://pandapower.readthedocs.io/en/stable/elements/load.html
# pandapower counts a load's power as positive when it is consumed, so the values
# go in exactly as given (no minus sign). In the D06 Gauss-Seidel notebook the same
# load was P2 = -2.566 pu, because there power *injected* into a bus counts as positive.
pp.create_load(net, bus=b2, p_mw=256.6, q_mvar=110.2, name="Load 2")


# BUS 3 
# Create a load bus, P=138.6MW, Q=45.2 Mvar
# https://pandapower.readthedocs.io/en/stable/elements/load.html
pp.create_load(net, bus=b3, p_mw=138.6, q_mvar=45.2, name="Load 3")

# inspect the load objects
net.load


In [ ]:
## Create lines
#https://pandapower.readthedocs.io/en/stable/elements/line.html
# NOTE: if x,r in PU then set length_km=Z_base to get from PU values to actual values.

# Calculate Z_Base = V/I = V^2/VI
VA_nom_MVA = 100 
Z_base = (V_nom_kV)**2/VA_nom_MVA
print ("Z_base = ",Z_base)

# Why the length_km trick works: pandapower multiplies "ohms per km" by "km".
# Entering the per-unit value as r_ohm_per_km and Z_base as length_km gives
#   total resistance = r_pu * Z_base  [ohms],
# which is exactly how you turn a per-unit impedance back into ohms.
# So the "length" is not a real distance here, just a unit conversion.
# c_nf_per_km (line capacitance) is set to almost zero because the exercise ignores it,
# and max_i_ka (the current rating) is set very high so it never limits anything.

# Line 1-2 0.02+j0.04 
pp.create_line_from_parameters(
    net, from_bus=b1, to_bus=b2,length_km=Z_base,
    r_ohm_per_km = 0.02, x_ohm_per_km = 0.04,
    c_nf_per_km = 1e-9, max_i_ka=1000, name="Line 1-2"
    )

# Line 2-3 0.0125+j0.025 
pp.create_line_from_parameters(
    net, from_bus=b2, to_bus=b3,length_km=Z_base,
    r_ohm_per_km = 0.0125, x_ohm_per_km = 0.025,
    c_nf_per_km = 1e-9, max_i_ka=1000, name="Line 2-3"
    )

# Line 1-3 0.01+j0.03 
pp.create_line_from_parameters(
    net, from_bus=b1, to_bus=b3,length_km=Z_base,
    r_ohm_per_km = 0.01, x_ohm_per_km = 0.03,
    c_nf_per_km = 1e-9, max_i_ka=1000, name="Line 1-3"
    )


In [ ]:
# inspect the line objects
net.line

In [ ]:
# Do a power flow
# https://pandapower.readthedocs.io/en/stable/powerflow/ac.html

pp.runpp(net, algorithm='gs');

In [ ]:
# Save the network to a json file
# https://pandapower.readthedocs.io/en/stable/file_io.html

pp.to_json(net,"ThreeBus.json")

In [ ]:
# View results on the buses
# vm_pu     = voltage magnitude in per unit     va_degree = voltage angle
# p_mw, q_mvar = power taken out at the bus (positive = consumed, negative = supplied).
# Bus 1 shows negative values because the slack bus is where power enters the network.
net.res_bus

In [ ]:
# View results on the lines
# p_from_mw, q_from_mvar = power entering the line at its "from" bus
# p_to_mw,   q_to_mvar   = power at the "to" end (negative = leaving the line there)
# pl_mw,     ql_mvar     = losses in the line (the difference between the two ends)
# A negative p_from_mw means the power actually flows the other way, from "to" to "from".
net.res_line

How do the results compare to your Gauss-Seidel solution from D6?

In [ ]:
# Comparison with the Gauss-Seidel solution from D06_S02
import numpy as np

# Turn magnitude + angle into a complex per-unit voltage, the form used in D06
V_pp = net.res_bus.vm_pu * np.exp(1j * np.deg2rad(net.res_bus.va_degree))

# Converged values from the D06_S02 notebook (iteration 14)
V_gs = {1: 0.98001 - 0.05998j, 2: 1.00001 - 0.04999j}

for i in [1, 2]:
    print(f"Bus {i+1}:  pandapower V = {V_pp[i]:.5f}    Gauss-Seidel (D06) V = {V_gs[i]:.5f}")

# Power supplied by the slack bus (the external grid)
P1 = net.res_ext_grid.p_mw.iloc[0]
Q1 = net.res_ext_grid.q_mvar.iloc[0]
print(f"\nSlack bus:  S1 = {P1:.1f} {Q1:+.1f}j MVA")

# Line flows written the same way as S12, S23, S13 in D06
print()
for idx, row in net.res_line.iterrows():
    f = net.line.from_bus.at[idx] + 1
    t = net.line.to_bus.at[idx] + 1
    print(f"S{f}{t} = {row.p_from_mw:7.1f} {row.q_from_mvar:+6.1f}j MVA    "
          f"losses = {row.pl_mw:5.2f} {row.ql_mvar:+6.2f}j MVA")

**Answer.** pandapower and the D06 Gauss-Seidel loop land on the same solution, which is also the lecturer's answer on the D06_S02 slides:

| | pandapower | Gauss-Seidel (D06) |
|---|---|---|
| Bus 2 voltage | 0.98000 − j0.06000 pu (0.9818 pu at −3.50°) | 0.98001 − j0.05998 pu |
| Bus 3 voltage | 1.00000 − j0.05000 pu (1.0012 pu at −2.86°) | 1.00001 − j0.04999 pu |
| Slack bus power | 409.5 MW, +189.0 Mvar | 409.4 MW, printed as −189j |

- **The last-decimal differences** come from when each method stops. The D06 loop stopped once the voltages changed by less than 1e-5 per iteration; pandapower keeps going until the power at every bus balances to within 1e-8 MVA.
- **Line flows and losses match too:** S12 ≈ 199.5 + j84.0, S13 = 210.0 + j105.0, S23 = −65.6 − j43.2 MVA. The minus sign on S23 means power really flows from bus 3 to bus 2. Losses: 8.5 + j17.0 (line 1-2), 5.0 + j15.0 (line 1-3), 0.8 + j1.6 MVA (line 2-3).
- **The sign of Q at the slack bus:** equation (4) in D06 gives **P − jQ**, so the −189j printed there means Q = **+189 Mvar**. That is what pandapower reports: the slack bus supplies the loads' 155.4 Mvar plus the 33.6 Mvar the line reactances absorb (17.0 + 15.0 + 1.6).
- **Why use pandapower then?** The answer is the same, but you no longer write the equations or the iteration loop yourself, and the same few lines work for a network with thousands of buses.

## Plotting
Pandapower also provides the ability to plot the network - including line loadings etc. 
Use the ```pplot.simple_plot``` to view a schematic of the network (with arbitrary locations assigned).

In [ ]:
# Simple plotting
# https://pandapower.readthedocs.io/en/stable/plotting.html
import pandapower.plotting as pplot

# The buses have no coordinates yet, so simple_plot invents a layout for them
# (it uses the igraph package for that, which is in your GeoPower environment).
# plot_loads=True also draws the two loads.
pplot.simple_plot(net, plot_loads=True)

Now add the following geospatial data to the buses:

| Bus  | Longitude (x) | Latitude (y) |
|-----:|--------------:|-------------:|
| Bus 1 | -1.2763 | 51.7503   |
| Bus 2 |  -1.189      | 51.7236      |
| Bus 3 | -1.3269 | 51.6312  |

You can either edit the code above when you created the buses in the first place (and re-run the notebook), or you can edit the bus dataframe directly. Pandapower stores the geospatial data in geojson format within the ```geo``` column. The code below shows you how to create geodataframes for the buses and lines, and how to assign the geospatial data to the buses. Pandapower uses the WGS84 coordinate system (EPSG:4326) for geospatial data (e.g. longitude and latitude).

In [ ]:
import geopandas as gpd
import json
from shapely.geometry import Point, LineString, mapping

# -----------------------------------
# 1. Create bus GeoDataFrame
# -----------------------------------

bus_coords = {
    0: (-1.2763, 51.7503),
    1: (-1.1890, 51.7236),
    2: (-1.3269, 51.6312)
}

gdf_buses = gpd.GeoDataFrame(
    geometry=[Point(coords) for coords in bus_coords.values()],
    index=list(bus_coords.keys()),
    crs="EPSG:4326"
)

# -----------------------------------
# 2. Create line GeoDataFrame
# notice lines are lists of coordinates, here we assume a straight line between the buses, but you could also use a more complex routing geometries by providing more points.
# -----------------------------------

line_coords = {
    0: [bus_coords[0], bus_coords[1]],
    1: [bus_coords[1], bus_coords[2]],
    2: [bus_coords[0], bus_coords[2]]
}

gdf_lines = gpd.GeoDataFrame(
    geometry=[LineString(coords) for coords in line_coords.values()],
    index=list(line_coords.keys()),
    crs="EPSG:4326"
)

# -----------------------------------
# 3. Assign geometries to pandapower
# -----------------------------------

net.bus.loc[gdf_buses.index, "geo"] = gdf_buses.geometry.apply(
    lambda geom: json.dumps(mapping(geom))
)

net.line.loc[gdf_lines.index, "geo"] = gdf_lines.geometry.apply(
    lambda geom: json.dumps(mapping(geom))
)


## Plotting with plotly
Pandapower has added support for plotly plotting. This allows for interactive plots, including zooming and panning. Use the ```pplot.simple_plotly``` function to view the network in an interactive plot. You can also plot the power flow results using the pf_res_plotly function. 

Note: There appears to be a bug in the current version of the pandapower plotting library related to a reverse geolocated trying to assign an address to points. The code below bypasses this issue.

The documentation is here: [pandapower plotly plotting](https://pandapower.readthedocs.io/en/stable/plotting.html#plotting-with-plotly)

In [ ]:
import pandapower.plotting as pp_plot   # added: this and later cells call the plotting module "pp_plot"
import pandapower.plotting.plotly.traces as traces

# Temporary workaround for reverse-geocoding failure
traces._on_map_test = lambda x, y: True

fig = pp_plot.simple_plotly(
    net,
    on_map=True,
    use_line_geo=True,
    map_style="open-street-map",
    aspectratio=(1,1), figsize=1
    
)

In [ ]:
# Re-plot the data, including color coding of the power flow results using the pf_res_plotly function.
# Buses are coloured by voltage (vm_pu) and lines by loading (% of their current rating).
# climits_volt sets the ends of the voltage colour scale. Our voltages lie between
# about 0.98 and 1.05 pu, so a narrower range than the default (0.9-1.1) shows the differences.
# Note: every line will show ~0 % loading, because max_i_ka=1000 kA is far above the
# ~0.5 kA that actually flows. Try a realistic rating (e.g. max_i_ka=1) to see the line colours change.
fig = pp_plot.pf_res_plotly(
    net,
    on_map=True,
    use_line_geo=True,
    map_style="open-street-map",
    aspectratio=(1,1), figsize=1,
    climits_volt=(0.97, 1.05),
)

## Exporting results to a geodataframe
pandapower has removed support for direct export of results to a geodataframe. You can use the code below to create geodataframes.

In [ ]:
import geopandas as gpd
# create a GeoDataFrame
# Fixed: in your pandapower version (3.2) these arguments are called nodes= and branches=;
# newer versions renamed them buses= and lines=. Passing them by position, in the order
# (net, buses, lines), works with both versions.
bus_geojson = pp_plot.geo.dump_to_geojson(net, True)            # buses only
bus_gdf = gpd.GeoDataFrame.from_features(bus_geojson["features"], crs="EPSG:4326")
lines_geojson = pp_plot.geo.dump_to_geojson(net, False, True)   # lines only
lines_gdf = gpd.GeoDataFrame.from_features(lines_geojson["features"], crs="EPSG:4326")

You can then perform spatial analysis on the results if necessary. 


In [ ]:
# Line lengths from the map geometry.
# (Fixed: the starter had a typo, "lines.gdf", and multiplied degrees by 100.)
# The geometry is stored in degrees of longitude/latitude, which are not kilometres:
# around Oxford one degree of longitude is ~69 km and one degree of latitude ~111 km.
# So first project to a coordinate system measured in metres, EPSG:27700 (British National Grid).
lines_gdf["length_km"] = lines_gdf.to_crs(epsg=27700).length / 1000
lines_gdf[["name", "length_km"]]